# Stripe removal: I12 parameter comparisons

Compare sorting, Titarenko, Fourier-wavelet, combined stripe removal and Raven filtering on the same experimental acquisition. Residual norms use the complete original arrays, not the displayed subsamples.

Transferred from `zenodo-tests/test_prep/test_stripe.py` (SHA-256 `da3186c02332005a7b51c1a1be5350fb1fbefad608fd1e24eb6e532551603660`).
Data: [Zenodo record 17252190](https://zenodo.org/records/17252190).
The original numerical references and tolerances are retained. Figures support visual review;
regression checks are evaluated after all cases so failures retain their figures and measurements.

## Execution

Install `.[validation]` and download the archives with `.scripts/download_zenodo.py Zenodo`.
Set `HTTOMOLIBGPU_DATA_DIR` to use an existing archive directory. Run GPU notebooks serially:

```bash
python -m pytest --nbmake --nbmake-timeout=1800 --overwrite docs/source/validation/preprocessing/stripe_removal.ipynb
```

For a conda CUDA installation, set `CUDA_PATH` to the toolkit root containing
`include/cuda_runtime.h` if CuPy cannot locate the headers. Documentation renders
saved outputs without executing GPU computations.

## Environment and archive loading


In [ ]:
import gc
import hashlib
import json
import os
import platform
from importlib.metadata import version
from pathlib import Path
from math import isclose

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np
import pytest
from numpy.testing import assert_allclose

from httomolibgpu.prep.normalize import dark_flat_field_correction, minus_log

repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "httomolibgpu").is_dir()
    ),
    None,
)
configured_dir = os.environ.get("HTTOMOLIBGPU_DATA_DIR")
if configured_dir:
    data_dir = Path(configured_dir).expanduser()
elif repo_root is not None:
    data_dir = repo_root / "Zenodo"
else:
    raise FileNotFoundError("Set HTTOMOLIBGPU_DATA_DIR to the Zenodo archive directory")
props = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
print(
    json.dumps(
        {
            "python": platform.python_version(),
            "packages": {
                name: version(name)
                for name in (
                    "httomolibgpu",
                    "tomobar",
                    "astra-toolbox",
                    "numpy",
                    "matplotlib",
                )
            },
            "cupy": cp.__version__,
            "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
            "cuda_driver": cp.cuda.runtime.driverGetVersion(),
            "gpu": (
                props["name"].decode()
                if isinstance(props["name"], bytes)
                else props["name"]
            ),
        },
        indent=2,
    )
)
results = []


def force_clean_gpu_memory():
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()


def load_dataset(name):
    path = data_dir / (name + ".npz")
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}; run .scripts/download_zenodo.py first"
        )
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    print(
        json.dumps(
            {
                "dataset": path.name,
                "zenodo_record": "17252190",
                "sha256": digest.hexdigest(),
            },
            indent=2,
        )
    )
    with np.load(path) as archive:
        if name == "i12_dataset3":
            return tuple(
                cp.asarray(archive[key]) for key in ("proj1", "proj2", "flats", "darks")
            )
        return (
            cp.asarray(archive["projdata"]),
            archive["angles"].copy(),
            cp.asarray(archive["flats"]),
            cp.asarray(archive["darks"]),
        )


def plot_comparison(before, after, title, label="Negative log transmission"):
    before, after = cp.asnumpy(before), cp.asnumpy(after)
    residual = before - after
    lo, hi = np.percentile(before[np.isfinite(before)], [1, 99])
    limit = float(np.percentile(np.abs(residual[np.isfinite(residual)]), 99)) or 1
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    for ax, image, name, cmap, vmin, vmax in (
        (axes[0], before, "Before", "gray", lo, hi),
        (axes[1], after, "After", "gray", lo, hi),
        (axes[2], residual, "Removed component", "coolwarm", -limit, limit),
    ):
        im = ax.imshow(image, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
        ax.set(title=name, xlabel="Detector column", ylabel="Projection index")
        fig.colorbar(im, ax=ax, label=label)
    fig.suptitle(title)
    plt.show()
    plt.close(fig)


def check_results():
    print(json.dumps(results, indent=2))
    failed = [row for row in results if not row["passed"]]
    assert not failed, "Transferred regressions failed: " + json.dumps(failed, indent=2)
    print(f"All {len(results)} transferred cases passed.")

In [ ]:
from httomolibgpu.prep.stripe import (
    remove_stripe_based_sorting,
    remove_stripe_ti,
    remove_stripe_fw,
    remove_all_stripe,
    raven_filter,
)

dataset = load_dataset("i12_dataset4")
data_normalised = minus_log(
    dark_flat_field_correction(dataset[0], dataset[2], dataset[3])
)
# Slice before normalisation exactly as the original Raven cases did.
raven_input = minus_log(
    dark_flat_field_correction(
        dataset[0][:, 10:20, :], dataset[2][:, 10:20, :], dataset[3][:, 10:20, :]
    )
)
del dataset
force_clean_gpu_memory()
print("Full input:", data_normalised.shape, "Raven input:", raven_input.shape)

## Sorting

Original function: `test_remove_stripe_based_sorting_i12_dataset4` (3 parameter cases).
Check the Euclidean norm of the removed component using `math.isclose` with
`abs_tol=1e-4` (and its original default relative tolerance), float32 dtype and C layout.

Before/after images share a display scale. Each plot uses the middle detector row;
angle subsampling is only for display and does not alter the residual norm.


In [ ]:
input_data = data_normalised
cases = [
    {"parameters": {"size": 11, "dim": 1}, "expected_norm": 78.8627},
    {"parameters": {"size": 21, "dim": 1}, "expected_norm": 92.8653},
    {"parameters": {"size": 31, "dim": 1}, "expected_norm": 99.6979},
]
for case in cases:
    output = remove_stripe_based_sorting(cp.copy(input_data), **case["parameters"])
    observed = float(cp.linalg.norm((input_data - output).flatten()).get())
    passed = (
        isclose(observed, case["expected_norm"], abs_tol=1e-4)
        and output.dtype == np.float32
        and output.flags.c_contiguous
    )

    results.append(
        {
            "test": "test_remove_stripe_based_sorting_i12_dataset4",
            **case,
            "observed_norm": observed,
            "dtype": str(output.dtype),
            "c_contiguous": bool(output.flags.c_contiguous),
            "passed": bool(passed),
        }
    )
    print(json.dumps(results[-1], indent=2))
    row = input_data.shape[1] // 2
    stride = max(1, input_data.shape[0] // 1000)
    plot_comparison(
        input_data[::stride, row, :],
        output[::stride, row, :],
        "Sorting: " + str(case["parameters"]),
    )
    del output
    force_clean_gpu_memory()

## Titarenko

Original function: `test_remove_stripe_ti_i12_dataset4` (3 parameter cases).
Check the Euclidean norm of the removed component using `math.isclose` with
`abs_tol=1e-4` (and its original default relative tolerance), float32 dtype and C layout.

Before/after images share a display scale. Each plot uses the middle detector row;
angle subsampling is only for display and does not alter the residual norm.


In [ ]:
input_data = data_normalised
cases = [
    {"parameters": {"beta": 0.01}, "expected_norm": 322.5501},
    {"parameters": {"beta": 0.03}, "expected_norm": 173.1643},
    {"parameters": {"beta": 0.06}, "expected_norm": 128.8032},
]
for case in cases:
    output = remove_stripe_ti(cp.copy(input_data), **case["parameters"])
    observed = float(cp.linalg.norm((input_data - output).flatten()).get())
    passed = (
        isclose(observed, case["expected_norm"], abs_tol=1e-4)
        and output.dtype == np.float32
        and output.flags.c_contiguous
    )

    results.append(
        {
            "test": "test_remove_stripe_ti_i12_dataset4",
            **case,
            "observed_norm": observed,
            "dtype": str(output.dtype),
            "c_contiguous": bool(output.flags.c_contiguous),
            "passed": bool(passed),
        }
    )
    print(json.dumps(results[-1], indent=2))
    row = input_data.shape[1] // 2
    stride = max(1, input_data.shape[0] // 1000)
    plot_comparison(
        input_data[::stride, row, :],
        output[::stride, row, :],
        "Titarenko: " + str(case["parameters"]),
    )
    del output
    force_clean_gpu_memory()

## Fourier-wavelet

Original function: `test_remove_stripe_fw_i12_dataset4` (3 parameter cases).
Check the Euclidean norm of the removed component using `math.isclose` with
`abs_tol=1e-4` (and its original default relative tolerance), float32 dtype and C layout.

Before/after images share a display scale. Each plot uses the middle detector row;
angle subsampling is only for display and does not alter the residual norm.


In [ ]:
input_data = data_normalised
cases = [
    {"parameters": {"sigma": 0.01, "level": 7}, "expected_norm": 67.9479},
    {"parameters": {"sigma": 0.3, "level": 5}, "expected_norm": 68.80203},
    {"parameters": {"sigma": 1.0, "level": 10}, "expected_norm": 873.24664},
]
for case in cases:
    output = remove_stripe_fw(cp.copy(input_data), **case["parameters"])
    observed = float(cp.linalg.norm((input_data - output).flatten()).get())
    passed = (
        isclose(observed, case["expected_norm"], abs_tol=1e-4)
        and output.dtype == np.float32
        and output.flags.c_contiguous
    )

    results.append(
        {
            "test": "test_remove_stripe_fw_i12_dataset4",
            **case,
            "observed_norm": observed,
            "dtype": str(output.dtype),
            "c_contiguous": bool(output.flags.c_contiguous),
            "passed": bool(passed),
        }
    )
    print(json.dumps(results[-1], indent=2))
    row = input_data.shape[1] // 2
    stride = max(1, input_data.shape[0] // 1000)
    plot_comparison(
        input_data[::stride, row, :],
        output[::stride, row, :],
        "Fourier-wavelet: " + str(case["parameters"]),
    )
    del output
    force_clean_gpu_memory()

## Combined stripe removal

Original function: `test_remove_all_stripe_i12_dataset4` (4 parameter cases).
Check the Euclidean norm of the removed component using `math.isclose` with
`abs_tol=1e-4` (and its original default relative tolerance), float32 dtype and C layout.
The combined method also retains its explicit NaN check.
Before/after images share a display scale. Each plot uses the middle detector row;
angle subsampling is only for display and does not alter the residual norm.


In [ ]:
input_data = data_normalised
cases = [
    {
        "parameters": {
            "snr": 1.0,
            "la_size": 31,
            "sm_size": 10,
            "dim": 1,
            "normalize": True,
        },
        "expected_norm": 105.3542,
    },
    {
        "parameters": {
            "snr": 2.0,
            "la_size": 41,
            "sm_size": 17,
            "dim": 1,
            "normalize": True,
        },
        "expected_norm": 99.9182,
    },
    {
        "parameters": {
            "snr": 3.0,
            "la_size": 61,
            "sm_size": 21,
            "dim": 1,
            "normalize": True,
        },
        "expected_norm": 103.3776,
    },
    {
        "parameters": {
            "snr": 4.0,
            "la_size": 71,
            "sm_size": 31,
            "dim": 1,
            "normalize": True,
        },
        "expected_norm": 106.6767,
    },
]
for case in cases:
    output = remove_all_stripe(cp.copy(input_data), **case["parameters"])
    observed = float(cp.linalg.norm((input_data - output).flatten()).get())
    passed = (
        isclose(observed, case["expected_norm"], abs_tol=1e-4)
        and output.dtype == np.float32
        and output.flags.c_contiguous
    )
    passed = passed and not bool(cp.isnan(output).any())
    results.append(
        {
            "test": "test_remove_all_stripe_i12_dataset4",
            **case,
            "observed_norm": observed,
            "dtype": str(output.dtype),
            "c_contiguous": bool(output.flags.c_contiguous),
            "passed": bool(passed),
        }
    )
    print(json.dumps(results[-1], indent=2))
    row = input_data.shape[1] // 2
    stride = max(1, input_data.shape[0] // 1000)
    plot_comparison(
        input_data[::stride, row, :],
        output[::stride, row, :],
        "Combined stripe removal: " + str(case["parameters"]),
    )
    del output
    force_clean_gpu_memory()

## Raven

Original function: `test_raven_filter_i12_dataset4` (3 parameter cases).
Check the Euclidean norm of the removed component using `math.isclose` with
`abs_tol=1e-4` (and its original default relative tolerance), float32 dtype and C layout.

Before/after images share a display scale. Each plot uses the middle detector row;
angle subsampling is only for display and does not alter the residual norm.


In [ ]:
input_data = raven_input
cases = [
    {
        "parameters": {
            "pad_y": 20,
            "pad_x": 20,
            "pad_method": "edge",
            "uvalue": 20,
            "nvalue": 2,
            "vvalue": 4,
        },
        "expected_norm": 94.0996,
    },
    {
        "parameters": {
            "pad_y": 20,
            "pad_x": 20,
            "pad_method": "edge",
            "uvalue": 20,
            "nvalue": 4,
            "vvalue": 2,
        },
        "expected_norm": 86.3459,
    },
    {
        "parameters": {
            "pad_y": 20,
            "pad_x": 20,
            "pad_method": "edge",
            "uvalue": 20,
            "nvalue": 6,
            "vvalue": 5,
        },
        "expected_norm": 111.1377,
    },
]
for case in cases:
    output = raven_filter(cp.copy(input_data), **case["parameters"])
    observed = float(cp.linalg.norm((input_data - output).flatten()).get())
    passed = (
        isclose(observed, case["expected_norm"], abs_tol=1e-4)
        and output.dtype == np.float32
        and output.flags.c_contiguous
    )

    results.append(
        {
            "test": "test_raven_filter_i12_dataset4",
            **case,
            "observed_norm": observed,
            "dtype": str(output.dtype),
            "c_contiguous": bool(output.flags.c_contiguous),
            "passed": bool(passed),
        }
    )
    print(json.dumps(results[-1], indent=2))
    row = input_data.shape[1] // 2
    stride = max(1, input_data.shape[0] // 1000)
    plot_comparison(
        input_data[::stride, row, :],
        output[::stride, row, :],
        "Raven: " + str(case["parameters"]),
    )
    del output
    force_clean_gpu_memory()

In [ ]:
del data_normalised, raven_input, input_data
force_clean_gpu_memory()
check_results()